# Lesson 04 - Tool Use Design Pattern

In this lesson you will learn the **Tool Use** design pattern for AI agents using the Microsoft Agent Framework (Python). We cover:

- Defining function tools with the `@tool` decorator and typed parameters
- Providing tool schemas so the model knows what each tool does
- Controlling tool execution with `approval_mode`
- Returning **structured output** via Pydantic models and `response_format`

The scenario is a **travel booking agent** that can look up destinations, check availability, and retrieve flight information.

## Setup

In [1]:
! pip install agent-framework agent-framework-foundry azure-ai-projects azure-identity python-dotenv -U -q

In [2]:
import logging

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework_foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
load_dotenv("../../.env")


In [3]:
# Create the Azure AI Foundry provider
client = FoundryChatClient(
    credential=AzureCliCredential(),
    model=os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o"),
)


## Defining Tools with the @tool Decorator

The `@tool` decorator turns a plain Python function into a tool that an agent can call.
Key points:

- The **docstring** becomes the tool description the model sees.
- **Type annotations** (including `Annotated` with descriptions) define the tool schema.
- `approval_mode` controls whether the user must approve each call before it executes.

In [4]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creating an Agent with Multiple Tools

Pass all three tools to the client so the model can invoke whichever ones it needs to answer the user's question.

In [5]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

Here are our current destinations and availability:

- Barcelona — Available (3 spots left)
- Paris — Available (no limit shown)
- Berlin — Sold out
- Tokyo — Available (1 spot left)
- Sydney — Available
- New York City — Available

Would you like me to:
- hold a spot for you (tip: holds are typically limited time),
- or check specific travel dates and the number of travelers?
If you provide your origin city/airport, I can also pull flight options for your top choices (e.g., Barcelona, Tokyo, Paris, NYC).


## Structured Output with Tools

By setting `response_format` to a Pydantic model, the agent is forced to return a well-typed JSON object instead of free-form text. This is useful when downstream code needs to consume the result programmatically.

In [6]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

Here are warm European options from your request, with current availability and flight details.

Destinations (from London Heathrow)
- Barcelona (Spain) — Available (3 spots left)
- Paris (France) — Available (no direct flights listed in our system)
- Berlin (Germany) — Sold out

Flight options from LHR
- LHR to BCN (Barcelona)
  - BA 2042
  - Departs: 08:30
  - Arrives: 11:45
  - Price: $350
  - Status: Direct flight available

- LHR to PAR (Paris)
  - No direct flights detected from LHR to PAR in the current data

Notes and recommendations
- Among the European options listed, Barcelona is the warmest option with an available direct flight.
- Paris is available but would require a connecting itinerary (not shown in the current data).
- Berlin is sold out right now.

Next steps (how would you like to proceed?)
- Book Barcelona now using the direct flight above, or hold a spot (3 left) if you’d like.
- If you prefer Paris, I can look for indirect routes via another airport (e.g., LHR → 

## Structured Output with Pydantic Validation

The previous cell lets the LLM return text, but nothing guarantees the shape is correct.
Here we pass `response_format=TravelPlanStrict` via `default_options` so the provider enforces the Pydantic schema server-side,
then use `response.value` (which calls `model_validate_json()` internally) to get
a fully-typed Python object instead of raw text.

**Important:** Azure AI strict mode requires `additionalProperties: false` on every object definition in the JSON schema.
Pydantic doesn't emit this by default, so we add `model_config = ConfigDict(json_schema_extra={"additionalProperties": False})`
to each model class.

In [7]:
# ---------- Pydantic-validated structured output ----------
#
# Key differences from the cell above:
#   1. response_format=TravelPlanStrict is passed via default_options
#      → tells the LLM provider to constrain output to the JSON schema
#   2. response.value auto-parses the JSON into a TravelPlanStrict instance
#      → raises ValidationError if the LLM returns malformed data
#   3. We iterate typed BookingRecommendationStrict objects, not raw strings
#
# IMPORTANT: Azure AI strict mode requires additionalProperties=false on
# every object in the schema. Pydantic doesn't emit this by default, so we
# set json_schema_extra on each model to satisfy the constraint.

import json as _json
from pydantic import BaseModel, ConfigDict, ValidationError


class BookingRecommendationStrict(BaseModel):
    """A single flight recommendation with strict schema."""
    model_config = ConfigDict(json_schema_extra={"additionalProperties": False})

    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlanStrict(BaseModel):
    """A collection of recommendations with strict schema."""
    model_config = ConfigDict(json_schema_extra={"additionalProperties": False})

    recommendations: list[BookingRecommendationStrict]


validated_agent = client.as_agent(
    name="ValidatedTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results "
        "matching the TravelPlanStrict JSON schema."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
    # response_format enforces the Pydantic schema at the LLM level
    default_options={"response_format": TravelPlanStrict},
)

response = await validated_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)

# --- Raw text (what the LLM actually returned) ---
print("Raw response text:")
print(response.text)
print()

# --- Parsed Pydantic model via .value ---
# .value calls TravelPlanStrict.model_validate_json(response.text) internally
# and caches the result — raises ValidationError on schema mismatch
try:
    plan: TravelPlanStrict = response.value
except ValidationError as e:
    print(f"Validation failed: {e}")
    raise

print(f"Received {len(plan.recommendations)} recommendation(s):\n")
for i, rec in enumerate(plan.recommendations, 1):
    print(f"  #{i} {rec.destination}")
    print(f"     Available:      {rec.available}")
    print(f"     Flight Details: {rec.flight_details}")
    print(f"     Estimated Cost: ${rec.estimated_cost}")
    print()

# --- Demonstrate Pydantic model features ---
print("Model dump (dict):")
print(_json.dumps(plan.model_dump(), indent=2))
print()
print("JSON schema sent to Azure (note additionalProperties on every object):")
print(_json.dumps(TravelPlanStrict.model_json_schema(), indent=2))


Raw response text:
{"recommendations":[{"destination":"Barcelona","available":true,"flight_details":"BA 2042, Departs 08:30, Arrives 11:45","estimated_cost":350}]}

Received 1 recommendation(s):

  #1 Barcelona
     Available:      True
     Flight Details: BA 2042, Departs 08:30, Arrives 11:45
     Estimated Cost: $350

Model dump (dict):
{
  "recommendations": [
    {
      "destination": "Barcelona",
      "available": true,
      "flight_details": "BA 2042, Departs 08:30, Arrives 11:45",
      "estimated_cost": 350
    }
  ]
}

JSON schema sent to Azure (note additionalProperties on every object):
{
  "$defs": {
    "BookingRecommendationStrict": {
      "additionalProperties": false,
      "description": "A single flight recommendation with strict schema.",
      "properties": {
        "destination": {
          "title": "Destination",
          "type": "string"
        },
        "available": {
          "title": "Available",
          "type": "boolean"
        },
        "fligh

## Tool Approval Patterns

The `approval_mode` parameter on `@tool` controls whether tool calls require human approval before executing:

| Mode | Behaviour |
|---|---|
| `"never_require"` | Tool runs automatically — no user confirmation needed. |
| `"always_require"` | Every call must be approved by the user before it executes. |

Use `"always_require"` for tools that have side-effects (e.g. booking a flight, charging a credit card) so a human stays in the loop.

In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

Tool name: book_flight
Approval mode: always_require


## Summary

In this lesson you learned how to:

1. **Define tools** using the `@tool` decorator with typed parameters and docstrings that serve as the tool schema.
2. **Compose multiple tools** so the agent can call them in sequence to answer complex queries.
3. **Return structured output** by passing a Pydantic model as `response_format`.
4. **Control tool approval** with `approval_mode` to keep a human in the loop for sensitive operations.

These patterns form the foundation for building reliable, production-ready agents that can interact with external systems safely.